<a href="https://colab.research.google.com/github/yunussfr/MSIDdetection/blob/main/Deit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
=============================================================
 Monkeypox Cilt Görüntüsü Sınıflandırıcı — DeiT Mimarisi
=============================================================
Özellikler:
  ✅ DeiT (Data-Efficient Image Transformer) backbone
  ✅ Özel sınıflandırıcı head: BN → Dropout → FC → Dropout → FC
  ✅ TEK KLASÖR → otomatik train/val/test split (%70/%15/%15)
  ✅ Stratified split: her sınıf oranı korunur
  ✅ Sınıf dengesizliği için ağırlıklı CrossEntropy kaybı
  ✅ WeightedRandomSampler: azınlık sınıflarına fazla örnekleme
  ✅ Cosine Annealing ile LR azaltma (yüksekten düşüğe)
  ✅ Early Stopping (overfitting başlayınca durur)
  ✅ En iyi modeli otomatik kaydeder (best_model.pth)
  ✅ Az veri için güçlü augmentation (TrivialAugmentWide)
=============================================================
Kurulum:
  pip install torch torchvision timm scikit-learn tqdm matplotlib
=============================================================
Veri klasör yapısı (Kaggle'dan indirilen — AYRILMAMIŞ):
  data/
  ├── Monkeypox/
  │   ├── img001.jpg
  │   └── ...
  ├── Chickenpox/
  ├── Measles/
  ├── Normal/
  └── ...

  ⚠️  train/val/test ayrımı YOKSA → DATA_DIR'i direkt göster,
      kod otomatik böler. Zaten ayrılmışsa SPLIT_MODE = "presplit"
      yap ve TRAIN_DIR / VAL_DIR / TEST_DIR ayarla.
=============================================================
"""

import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import timm
from tqdm import tqdm


# ─────────────────────────────────────────────
# 1. AYARLAR
# ─────────────────────────────────────────────
class Config:
    # ── Veri ─────────────────────────────────
    # "auto"      → Tek klasör, kod train/val/test'e böler
    # "presplit"  → Zaten ayrılmışsa bu modu seç
    SPLIT_MODE = "auto"

    DATA_DIR  = "/content/drive/MyDrive/Monkeypox Skin Image Dataset"           # SPLIT_MODE="auto" → sınıf klasörleri burada
    TRAIN_DIR =  "/content/drive/MyDrive/train_data"    # SPLIT_MODE="presplit" için
    VAL_DIR   = "/content/drive/MyDrive/val_data"       # SPLIT_MODE="presplit" için
    TEST_DIR  = "./content/drive/MyDrive/test_data"      # SPLIT_MODE="presplit" için

    # Auto-split oranları (toplamı 1.0 olmalı)
    TRAIN_RATIO = 0.70
    VAL_RATIO   = 0.15
    TEST_RATIO  = 0.15
    SPLIT_SEED  = 42               # Tekrarlanabilir split

    OUTPUT_DIR  = "./outputs"

    # ── Model ────────────────────────────────
    MODEL_NAME  = "deit_small_patch16_224"
    # Alternatifler: "deit_tiny_patch16_224" | "deit_base_patch16_224"
    PRETRAINED  = True
    NUM_CLASSES = None             # Otomatik tespit edilir

    # Özel head ayarları (az veri → Dropout kritik!)
    HEAD_HIDDEN  = 256             # Ara katman nöronu
    HEAD_DROPOUT = 0.4             # Dropout oranı (0.3–0.5 arası)

    # ── Eğitim ───────────────────────────────
    EPOCHS      = 50
    BATCH_SIZE  = 16               # Az veri → küçük batch daha stabil
    NUM_WORKERS = 4
    IMG_SIZE    = 224

    # ── Öğrenme hızı ─────────────────────────
    LR_START    = 5e-4             # Az veri → daha küçük LR ile başla
    LR_MIN      = 1e-6
    WEIGHT_DECAY= 1e-4

    # ── Freeze stratejisi (transfer learning) ─
    # Az veri varken backbone'u dondurup sadece head'i eğitmek iyi başlangıç
    # Kaç epoch sonra backbone'u çöz? 0 = hiç dondurma
    FREEZE_EPOCHS = 5              # İlk 5 epoch sadece head eğitilir

    # ── Early Stopping ───────────────────────
    PATIENCE  = 8
    MIN_DELTA = 1e-4

    # ── Cihaz ────────────────────────────────
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ─────────────────────────────────────────────
# 2. VERİ ARTIRMA
# ─────────────────────────────────────────────
def build_transforms(img_size: int):
    """
    Eğitim: TrivialAugmentWide + ekstra jitter (az veri için güçlü)
    Val/Test: sadece resize + center crop + normalize
    """
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        # TrivialAugmentWide: otomatik olarak en iyi augmentation seçer
        transforms.TrivialAugmentWide(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3,
                               saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
        # Random Erasing: görüntünün bir parçasını siler (overfitting'e karşı)
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    ])

    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])

    return train_tf, val_tf


# ─────────────────────────────────────────────
# 3. VERİ BÖLME & YÜKLEYİCİLER
# ─────────────────────────────────────────────
def stratified_split(targets, train_r, val_r, seed):
    """
    Tüm indeksleri stratified şekilde train / val / test'e böler.
    Her sınıfın oranı tüm split'lerde korunur.
    """
    indices = np.arange(len(targets))
    targets = np.array(targets)

    # Önce train / temp ayır
    train_idx, temp_idx = train_test_split(
        indices, train_size=train_r,
        stratify=targets, random_state=seed
    )
    # Sonra temp'i val / test olarak eşit böl
    val_ratio_of_temp = val_r / (1.0 - train_r)
    val_idx, test_idx = train_test_split(
        temp_idx, train_size=val_ratio_of_temp,
        stratify=targets[temp_idx], random_state=seed
    )
    return train_idx, val_idx, test_idx


import numpy as np
import torch

def compute_class_weights(targets, num_classes, smoothing_power=0.5) -> torch.Tensor:
    """
    Eğitim seti üzerinden sınıf ağırlıkları hesaplar ve azınlık sınıflarındaki aşırı cezaları yumuşatır.

    Parametreler:
        targets: Gerçek etiketlerin listesi veya dizisi
        num_classes: Toplam sınıf sayısı
        smoothing_power: 0.0 ile 1.0 arasında bir değer.
                         1.0 -> Eski sert cezalar (Orijinal hali)
                         0.5 -> Karekök yumuşatması (Tavsiye edilen dengeli hal)
                         0.0 -> Ağırlıklandırma yok (Tüm sınıflar 1 olur)
    """
    targets = np.array(targets)
    class_counts = np.bincount(targets, minlength=num_classes)
    total = len(targets)

    print("\n📊 Eğitim Seti Sınıf Dağılımı:")
    for i, cnt in enumerate(class_counts):
        print(f"   Sınıf {i:2d}: {cnt:5d} örnek  ({cnt/total*100:.1f}%)")

    # 1. Klasik (Sert) Ağırlıkları Hesapla
    base_weights = total / (num_classes * class_counts.astype(float) + 1e-8)

    # 2. Ağırlıkları Yumuşat (Üstel Fonksiyon ile)
    smoothed_weights = np.power(base_weights, smoothing_power)

    # 3. Normalizasyon (Çok Önemli!)
    # Ağırlıkların genel ortalamasını 1'e çekerek, genel Hata (Loss) büyüklüğünün
    # değişmesini ve Öğrenme Hızının (LR) bozulmasını engelliyoruz.
    smoothed_weights = smoothed_weights / np.mean(smoothed_weights)

    wt = torch.FloatTensor(smoothed_weights)
    print(f"⚖️  Yumuşatılmış Ağırlıklar: {wt.numpy().round(3)}")
    return wt


def build_dataloaders(cfg: Config):
    train_tf, val_tf = build_transforms(cfg.IMG_SIZE)

    if cfg.SPLIT_MODE == "auto":
        # ── Tek klasör: tamamını yükle, sonra böl ──
        # Önce transform'suz yükle (indeks için)
        full_ds = datasets.ImageFolder(cfg.DATA_DIR)
        cfg.NUM_CLASSES = len(full_ds.classes)

        print(f"\n🗂️  Toplam sınıf : {cfg.NUM_CLASSES} → {full_ds.classes}")
        print(f"   Toplam görüntü: {len(full_ds)}")

        # Stratified split
        train_idx, val_idx, test_idx = stratified_split(
            full_ds.targets,
            cfg.TRAIN_RATIO, cfg.VAL_RATIO,
            cfg.SPLIT_SEED
        )
        print(f"\n✂️  Split sonucu:")
        print(f"   Train : {len(train_idx):4d} görüntü")
        print(f"   Val   : {len(val_idx):4d} görüntü")
        print(f"   Test  : {len(test_idx):4d} görüntü")

        # Her split için ayrı transform uygula
        train_ds_raw = datasets.ImageFolder(cfg.DATA_DIR, transform=train_tf)
        val_ds_raw   = datasets.ImageFolder(cfg.DATA_DIR, transform=val_tf)

        train_ds = Subset(train_ds_raw, train_idx)
        val_ds   = Subset(val_ds_raw,   val_idx)
        test_ds  = Subset(val_ds_raw,   test_idx)

        # Sınıf ağırlıkları → sadece train üzerinden
        train_targets = np.array(full_ds.targets)[train_idx]
        class_names   = full_ds.classes

    else:
        # ── Önceden ayrılmış klasörler ──────────
        train_ds_raw = datasets.ImageFolder(cfg.TRAIN_DIR, transform=train_tf)
        val_ds_raw   = datasets.ImageFolder(cfg.VAL_DIR,   transform=val_tf)
        test_ds_raw  = datasets.ImageFolder(cfg.TEST_DIR,  transform=val_tf)

        cfg.NUM_CLASSES = len(train_ds_raw.classes)
        train_ds = train_ds_raw
        val_ds   = val_ds_raw
        test_ds  = test_ds_raw
        train_targets = np.array(train_ds_raw.targets)
        class_names   = train_ds_raw.classes

        print(f"\n🗂️  Sınıflar: {class_names}")

    # Sınıf ağırlıkları
    class_weights = compute_class_weights(train_targets, cfg.NUM_CLASSES)

    # WeightedRandomSampler (sadece train için)
    sample_weights = class_weights[train_targets]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE,
        sampler=sampler,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(cfg.DEVICE == "cuda"),
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(cfg.DEVICE == "cuda"),
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(cfg.DEVICE == "cuda"),
    )

    return train_loader, val_loader, test_loader, class_names, class_weights


# ─────────────────────────────────────────────
# 4. MODEL — Özel Sınıflandırıcı Head
# ─────────────────────────────────────────────
def build_model(cfg: Config) -> nn.Module:
    """
    DeiT backbone + özel head:
      [CLS token çıktısı]
          ↓
      BatchNorm1d          ← aktivasyonları normalize eder
          ↓
      Dropout(p)           ← overfitting'e karşı ilk bariyer
          ↓
      Linear(feat → hidden) + GELU
          ↓
      Dropout(p)           ← ikinci bariyer
          ↓
      Linear(hidden → num_classes)

    Az veri + pretrained backbone kombinasyonu için
    bu yapı standart tek-FC head'den daha sağlam çalışır.
    """
    # Backbone'u num_classes=0 ile yükle → ham feature çıkarır
    backbone = timm.create_model(
        cfg.MODEL_NAME,
        pretrained=cfg.PRETRAINED,
        num_classes=0,   # classifier'ı kaldır
    )
    feat_dim = backbone.num_features  # DeiT-small: 384

    # Özel head
    head = nn.Sequential(
        nn.BatchNorm1d(feat_dim),
        nn.Dropout(p=cfg.HEAD_DROPOUT),
        nn.Linear(feat_dim, cfg.HEAD_HIDDEN),
        nn.GELU(),
        nn.Dropout(p=cfg.HEAD_DROPOUT),
        nn.Linear(cfg.HEAD_HIDDEN, cfg.NUM_CLASSES),
    )

    # Ağırlık başlatma: head için Kaiming normal
    for layer in head:
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    # Backbone + head birleştir
    class DeiTWithCustomHead(nn.Module):
        def __init__(self, backbone, head):
            super().__init__()
            self.backbone = backbone
            self.head     = head

        def forward(self, x):
            features = self.backbone(x)   # (B, feat_dim)
            return self.head(features)    # (B, num_classes)

        def freeze_backbone(self):
            for p in self.backbone.parameters():
                p.requires_grad = False
            print("🔒 Backbone donduruldu — sadece head eğitilecek")

        def unfreeze_backbone(self):
            for p in self.backbone.parameters():
                p.requires_grad = True
            print("🔓 Backbone çözüldü — tüm model eğitilecek")

    model = DeiTWithCustomHead(backbone, head).to(cfg.DEVICE)

    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n🤖 Model   : {cfg.MODEL_NAME}")
    print(f"   Backbone feature dim : {feat_dim}")
    print(f"   Head hidden dim      : {cfg.HEAD_HIDDEN}")
    print(f"   Head dropout         : {cfg.HEAD_DROPOUT}")
    print(f"   Toplam parametre     : {total:,}")
    print(f"   Eğitilebilir param   : {trainable:,}")

    return model


# ─────────────────────────────────────────────
# 5. EĞİTİM DÖNGÜSÜ
# ─────────────────────────────────────────────

class EarlyStopping:
    """Val loss iyileşmezse eğitimi durdurur."""
    def __init__(self, patience: int, min_delta: float):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = float("inf")
        self.stop      = False

    def step(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f"   ⚠️  EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def run_epoch(model, loader, criterion, optimizer, device, is_train: bool):
    """Tek epoch çalıştırır, (loss, accuracy) döner."""
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    desc = "  Eğitim " if is_train else "  Val/Test"
    with torch.set_grad_enabled(is_train):
        for imgs, labels in tqdm(loader, desc=desc, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total


def train(cfg: Config):
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    best_path = os.path.join(cfg.OUTPUT_DIR, "best_model.pth")

    # Veri
    train_loader, val_loader, test_loader, class_names, class_weights = \
        build_dataloaders(cfg)

    # Model
    model = build_model(cfg)

    # Başlangıçta backbone'u dondur (az veri için önerilen)
    if cfg.FREEZE_EPOCHS > 0:
        model.freeze_backbone()

    # Kayıp: ağırlıklı CrossEntropy
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(cfg.DEVICE))

    # Optimizer: sadece requires_grad=True olanları alır
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LR_START,
        weight_decay=cfg.WEIGHT_DECAY,
    )

    # LR Scheduler: Cosine — başta yüksek, sonlara doğru LR_MIN'e iner
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.EPOCHS, eta_min=cfg.LR_MIN
    )

    early_stop = EarlyStopping(cfg.PATIENCE, cfg.MIN_DELTA)

    history = {k: [] for k in
               ["train_loss", "train_acc", "val_loss", "val_acc", "lr"]}
    best_val_acc = 0.0
    best_weights = None
    best_epoch   = 0

    print("\n" + "="*60)
    print(" EĞİTİM BAŞLIYOR")
    print("="*60)

    for epoch in range(1, cfg.EPOCHS + 1):

        # Backbone çözme: FREEZE_EPOCHS geçince optimizer'ı güncelle
        if cfg.FREEZE_EPOCHS > 0 and epoch == cfg.FREEZE_EPOCHS + 1:
            model.unfreeze_backbone()
            # Optimizer'a backbone parametrelerini ekle (daha düşük LR)
            optimizer.add_param_group({
                "params": model.backbone.parameters(),
                "lr": cfg.LR_START * 0.1,   # backbone için 10× düşük LR
            })
            print(f"   Backbone parametreleri optimizer'a eklendi "
                  f"(LR={cfg.LR_START*0.1:.1e})")

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\n📅 Epoch {epoch}/{cfg.EPOCHS}  |  LR: {current_lr:.2e}")
        print("-" * 45)
        t0 = time.time()

        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, cfg.DEVICE, True)
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, cfg.DEVICE, False)

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        elapsed = time.time() - t0
        print(f"   Train → Loss: {train_loss:.4f}  Acc: {train_acc*100:.2f}%")
        print(f"   Val   → Loss: {val_loss:.4f}  Acc: {val_acc*100:.2f}%")
        print(f"   Süre  : {elapsed:.1f}s")

        # En iyi modeli kaydet
        if val_acc > best_val_acc + cfg.MIN_DELTA:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            best_epoch   = epoch
            torch.save({
                "epoch":       epoch,
                "model_state": best_weights,
                "val_acc":     best_val_acc,
                "class_names": class_names,
                "model_name":  cfg.MODEL_NAME,
                "head_hidden": cfg.HEAD_HIDDEN,
                "head_dropout":cfg.HEAD_DROPOUT,
            }, best_path)
            print(f"   ✅ En iyi model kaydedildi! "
                  f"Val Acc: {best_val_acc*100:.2f}%")

        # Early Stopping
        if early_stop.step(val_loss):
            print(f"\n🛑 Early Stopping aktif (epoch {epoch}). Duruyorum.")
            break

    print("\n" + "="*60)
    print(f" EĞİTİM BİTTİ — En iyi Val Acc: {best_val_acc*100:.2f}%"
          f"  (Epoch {best_epoch})")
    print(f" Kaydedilen model: {best_path}")
    print("="*60)

    # En iyi ağırlıkları geri yükle
    model.load_state_dict(best_weights)

    # Final değerlendirme: TEST seti üzerinde
    print("\n🔬 TEST SETİ DEĞERLENDİRMESİ (en iyi model):")
    evaluate(model, test_loader, class_names, cfg.DEVICE, cfg.OUTPUT_DIR)
    plot_history(history, cfg.OUTPUT_DIR)

    return model, history


# ─────────────────────────────────────────────
# 6. SONUÇ DEĞERLENDİRME
# ─────────────────────────────────────────────

def evaluate(model, loader, class_names, device, output_dir="./outputs"):
    """Precision / Recall / F1 raporu + Confusion Matrix."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="  Test"):
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print("\n📋 Sınıflandırma Raporu:")
    print(classification_report(
        all_labels, all_preds, target_names=class_names, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    n  = len(class_names)
    fig, ax = plt.subplots(figsize=(max(6, n*1.2), max(5, n)))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_title("Confusion Matrix — Test Seti")
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
    plt.tight_layout()
    path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"   📊 Confusion matrix → {path}")


def plot_history(history: dict, output_dir: str):
    """Loss, Accuracy ve LR grafiklerini kaydeder."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"],   label="Val")
    axes[0].set_title("Kayıp (Loss)"); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs, [a*100 for a in history["train_acc"]], label="Train")
    axes[1].plot(epochs, [a*100 for a in history["val_acc"]],   label="Val")
    axes[1].set_title("Doğruluk (%)"); axes[1].set_xlabel("Epoch")
    axes[1].legend(); axes[1].grid(True)

    axes[2].semilogy(epochs, history["lr"])
    axes[2].set_title("Öğrenme Hızı"); axes[2].set_xlabel("Epoch")
    axes[2].grid(True)

    plt.suptitle("DeiT Eğitim Geçmişi — Monkeypox", fontsize=13)
    plt.tight_layout()
    path = os.path.join(output_dir, "training_history.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"   📈 Eğitim grafiği → {path}")


# ─────────────────────────────────────────────
# 7. TAHMİN (tek görüntü)
# ─────────────────────────────────────────────

def predict_image(image_path: str,
                  checkpoint_path: str = "outputs/best_model.pth"):
    """
    Kaydedilen en iyi modelle tek görüntü tahmin eder.
    Kullanım: predict_image("cilt_foto.jpg")
    """
    from PIL import Image

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    class_names  = ckpt["class_names"]
    model_name   = ckpt["model_name"]
    head_hidden  = ckpt.get("head_hidden",  256)
    head_dropout = ckpt.get("head_dropout", 0.4)
    num_classes  = len(class_names)

    # Modeli yeniden kur
    backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
    feat_dim = backbone.num_features
    head = nn.Sequential(
        nn.BatchNorm1d(feat_dim),
        nn.Dropout(head_dropout),
        nn.Linear(feat_dim, head_hidden),
        nn.GELU(),
        nn.Dropout(head_dropout),
        nn.Linear(head_hidden, num_classes),
    )

    class M(nn.Module):
        def __init__(self): super().__init__(); self.backbone=backbone; self.head=head
        def forward(self, x): return self.head(self.backbone(x))

    m = M()
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    _, val_tf = build_transforms(224)
    img = Image.open(image_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0)

    with torch.no_grad():
        probs = torch.softmax(m(tensor), dim=1)[0]

    print(f"\n🔍 Tahmin: {image_path}")
    for cls, p in sorted(zip(class_names, probs.numpy()), key=lambda x: -x[1]):
        bar = "█" * int(p * 30)
        print(f"   {cls:20s}: {p*100:5.1f}%  {bar}")

    best = class_names[probs.argmax()]
    print(f"\n   → Tahmin: {best}  (güven: {probs.max()*100:.1f}%)")
    return best, probs.numpy()


# ─────────────────────────────────────────────
# 8. ANA ÇALIŞMA NOKTASI
# ─────────────────────────────────────────────

if __name__ == "__main__":
    cfg = Config()

    print("🖥️  Cihaz     :", cfg.DEVICE)
    print("📁 Veri modu  :", cfg.SPLIT_MODE)
    if cfg.DEVICE == "cuda":
        print("   GPU       :", torch.cuda.get_device_name(0))

    model, history = train(cfg)

    # Tek görüntü tahmini (opsiyonel):
    # predict_image("cilt_foto.jpg")

🖥️  Cihaz     : cuda
📁 Veri modu  : auto
   GPU       : Tesla T4

🗂️  Toplam sınıf : 4 → ['Chickenpox', 'Measles', 'Monkeypox', 'Normal']
   Toplam görüntü: 770

✂️  Split sonucu:
   Train :  539 görüntü
   Val   :  115 görüntü
   Test  :  116 görüntü

📊 Eğitim Seti Sınıf Dağılımı:
   Sınıf  0:    75 örnek  (13.9%)
   Sınıf  1:    64 örnek  (11.9%)
   Sınıf  2:   195 örnek  (36.2%)
   Sınıf  3:   205 örnek  (38.0%)
⚖️  Yumuşatılmış Ağırlıklar: [1.209 1.309 0.75  0.731]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



🤖 Model   : deit_small_patch16_224
   Backbone feature dim : 384
   Head hidden dim      : 256
   Head dropout         : 0.4
   Toplam parametre     : 21,766,020
   Eğitilebilir param   : 21,766,020
🔒 Backbone donduruldu — sadece head eğitilecek

 EĞİTİM BAŞLIYOR

📅 Epoch 1/50  |  LR: 5.00e-04
---------------------------------------------


  Eğitim :   0%|          | 0/34 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


   Train → Loss: 2.0334  Acc: 36.55%
   Val   → Loss: 0.7627  Acc: 67.83%
   Süre  : 9.4s
   ✅ En iyi model kaydedildi! Val Acc: 67.83%

📅 Epoch 2/50  |  LR: 5.00e-04
---------------------------------------------


...

   Train → Loss: 0.1496  Acc: 96.10%
   Val   → Loss: 0.5418  Acc: 93.91%
   Süre  : 10.1s
   ⚠️  EarlyStopping: 6/8

📅 Epoch 25/50  |  LR: 2.66e-04
---------------------------------------------


   Train → Loss: 0.1315  Acc: 96.85%
   Val   → Loss: 0.5469  Acc: 94.78%
   Süre  : 8.3s
   ⚠️  EarlyStopping: 7/8

📅 Epoch 26/50  |  LR: 2.51e-04
---------------------------------------------


   Train → Loss: 0.1272  Acc: 96.29%
   Val   → Loss: 0.3284  Acc: 94.78%
   Süre  : 9.8s
   ⚠️  EarlyStopping: 8/8

🛑 Early Stopping aktif (epoch 26). Duruyorum.

 EĞİTİM BİTTİ — En iyi Val Acc: 95.65%  (Epoch 18)
 Kaydedilen model: ./outputs/best_model.pth

🔬 TEST SETİ DEĞERLENDİRMESİ (en iyi model):


  Test: 100%|██████████| 8/8 [00:01<00:00,  7.43it/s]



📋 Sınıflandırma Raporu:
              precision    recall  f1-score   support

  Chickenpox     0.7000    0.8750    0.7778        16
     Measles     0.8000    0.8571    0.8276        14
   Monkeypox     1.0000    0.8571    0.9231        42
      Normal     0.9111    0.9318    0.9213        44

    accuracy                         0.8879       116
   macro avg     0.8528    0.8803    0.8624       116
weighted avg     0.9008    0.8879    0.8909       116

   📊 Confusion matrix → ./outputs/confusion_matrix.png
   📈 Eğitim grafiği → ./outputs/training_history.png
